In [1]:
from pathlib import Path
import re

pdf_path = Path('/workspaces/qm2023-capstone-the-stat-servers/M1-assignment-description.pdf')
out_path = Path('/workspaces/qm2023-capstone-the-stat-servers/M1-assignment-description.extracted.md')

if not pdf_path.exists():
    raise FileNotFoundError(pdf_path)

reader = None
err_msgs = []

try:
    from pypdf import PdfReader
    reader = PdfReader(str(pdf_path))
except Exception as e:
    err_msgs.append(f'pypdf failed: {e}')

if reader is None:
    try:
        from PyPDF2 import PdfReader
        reader = PdfReader(str(pdf_path))
    except Exception as e:
        err_msgs.append(f'PyPDF2 failed: {e}')

if reader is None:
    raise RuntimeError('No PDF reader available. ' + ' | '.join(err_msgs))

def normalize_lines(text: str):
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [ln.rstrip() for ln in text.split('\n')]
    out = []
    prev_blank = False
    for ln in lines:
        ln = re.sub(r'[ \t]+', ' ', ln).rstrip()
        if ln == '':
            if not prev_blank:
                out.append('')
            prev_blank = True
        else:
            out.append(ln)
            prev_blank = False
    return out

md_pages = []
for i, page in enumerate(reader.pages, start=1):
    raw = page.extract_text() or ''
    lines = normalize_lines(raw)

    fixed = []
    for ln in lines:
        s = ln.strip()
        if re.match(r'^[\u2022\u25E6\u25AA\u25CF\-\*]\s+', s):
            s = re.sub(r'^[\u2022\u25E6\u25AA\u25CF]\s+', '- ', s)
            fixed.append(s)
        elif re.match(r'^\(?\d+[\)\.]\s+', s):
            fixed.append(s)
        else:
            # heading heuristic: short title-case/all-caps lines without ending punctuation
            if len(s) <= 80 and s and not s.endswith(('.', ';', ':', ',')):
                words = s.split()
                if len(words) <= 10 and (s.isupper() or sum(w[:1].isupper() for w in words) >= max(1, len(words)-1)):
                    if not s.startswith('#'):
                        fixed.append('### ' + s)
                        continue
            fixed.append(s)

    page_md = '\n'.join(fixed).strip()
    md_pages.append(f'## Page {i}\n\n' + page_md)

final_md = '\n\n'.join(md_pages).strip() + '\n'
out_path.write_text(final_md, encoding='utf-8')

print(f'Pages: {len(reader.pages)}')
print(f'Written: {out_path}')
print(final_md[:4000])

Pages: 5
Written: /workspaces/qm2023-capstone-the-stat-servers/M1-assignment-description.extracted.md
## Page 1

1 / 5
Milestone 1: Data Pipeline - Blackboard Assignment
### Description
Course: QM 2023: Statistics II / Data Analytics
Due: Wednesday, February 25, 2026 by 11:59 PM
Points: 50 (25% of capstone grade)
Submission: Team submission (one per team)
### Overview
Build your capstone project's foundational data infrastructure by integrating your primary dataset (REIT
returns by default, or your approved alternative) with supplementary economic/market data into a clean,
analysis-ready panel dataset.
Real-world context: Professional analysts spend 60-80% of their time on data engineering. A robust pipeline
ensures all subsequent analysis runs smoothly and reproducibly.
### What You're Submitting
1. GitHub Repository Link
Required: Submit the URL to your team's GitHub repository (created via GitHub Classroom).
### Example: https://github.com/Dr-Seagraves/qm2023-capstone-team-name
Your

In [2]:
import subprocess, sys
from pathlib import Path

repo_root = Path('/workspaces/qm2023-capstone-the-stat-servers')
cmd=[sys.executable,'code/fetch_integrate_supplementary_data.py','--start-date','2022-01-01','--end-date','2022-12-31']
res=subprocess.run(cmd,capture_output=True,text=True,cwd=repo_root)
print('exit',res.returncode)
print(res.stdout)
print(res.stderr)

for p in [
    'data/raw/supplementary_sources_selected.csv',
    'data/raw/supplementary_macro_monthly_raw.csv',
    'data/processed/supplementary_macro_monthly_features.csv',
    'data/final/supplementary_controls_panel.csv',
]:
    print(f'EXISTS {p} {(repo_root / p).exists()}')

exit 0
✓ Project structure verified at: /workspaces/qm2023-capstone-the-stat-servers
Saved selected source metadata: /workspaces/qm2023-capstone-the-stat-servers/data/raw/supplementary_sources_selected.csv
Saved raw supplementary panel: /workspaces/qm2023-capstone-the-stat-servers/data/raw/supplementary_macro_monthly_raw.csv
Saved processed supplementary panel: /workspaces/qm2023-capstone-the-stat-servers/data/processed/supplementary_macro_monthly_features.csv
Saved final supplementary controls panel: /workspaces/qm2023-capstone-the-stat-servers/data/final/supplementary_controls_panel.csv
Selected source rows from catalog: 3


EXISTS data/raw/supplementary_sources_selected.csv True
EXISTS data/raw/supplementary_macro_monthly_raw.csv True
EXISTS data/processed/supplementary_macro_monthly_features.csv True
EXISTS data/final/supplementary_controls_panel.csv True
